# Colab Experiment: Unlearnable Validation Protocol

Goal: select Unlearnable epsilon on a clean validation split, then evaluate the selected configuration once on the clean test split.


## 1. Setup Colab / GitHub repo

Clone the repo in a fresh Colab runtime and install dependencies.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ngocvuq4/adversarial-data-protection.git"
PROJECT_DIR = "adversarial-data-protection"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("src").exists():
    if not Path(PROJECT_DIR).exists():
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    os.chdir(PROJECT_DIR)

print("Working directory:", os.getcwd())
print("Installing: requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


## 2. Mount Google Drive

Datasets and results are read/written through the project folder in Drive.


In [ ]:
# Google Drive dataset/results paths.
# Your Drive folder is: MyDrive/adversarial-data-protection/
from pathlib import Path

USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/adversarial-data-protection"
DRIVE_DATA_ROOT = f"{DRIVE_PROJECT_DIR}/data"
DRIVE_RESULTS_DIR = f"{DRIVE_PROJECT_DIR}/results"

DATA_ROOT = "./data"
RESULTS_ROOT = "./results"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_ROOT = DRIVE_DATA_ROOT
        RESULTS_ROOT = DRIVE_RESULTS_DIR
        Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
        Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)
    except ImportError:
        print("Not running in Colab; using local ./data and ./results")

print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)


## 3. Validation Epsilon Ablation + Final Test

This cell uses CIFAR-10 train split for train/validation and reserves CIFAR-10 test split for final evaluation after selecting epsilon.


In [ ]:
import os
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, TensorDataset
import torchvision.transforms as T
from torchvision.datasets import CIFAR10

from scripts.run_experiment import setup_dirs, collect_clean_tensors, build_protected_tensors
from src.evaluation import compute_attack_success_rate, compute_linf, compute_psnr, compute_ssim
from src.models import evaluate, get_victim_resnet18, train_one_epoch
from src.techniques.unlearnable import generate_unlearnable_noise
from src.visualization import plot_before_after

# Validation protocol for hyperparameter selection.
# Test set is evaluated only after best_epsilon is selected from validation metrics.
SEED = 42
TRAIN_SUBSET_SIZE = 5000
VAL_SUBSET_SIZE = 1000
TEST_SUBSET_SIZE = 10000  # use 1000 for a faster dry run; use 10000 for final reporting
BATCH_SIZE = 128
EPSILON_VALUES = [0.01, 0.03, 0.05]
BASELINE_EPOCHS = 15
VICTIM_EPOCHS = 15
PGD_STEPS = 10
INNER_EPOCHS = 2
LEARNING_RATE = 0.1
WEIGHT_DECAY = 5e-4
MIN_PSNR = 30.0
MIN_SSIM = 0.90

RUN_NAME = f"train{TRAIN_SUBSET_SIZE}_val{VAL_SUBSET_SIZE}_seed{SEED}_base{BASELINE_EPOCHS}_victim{VICTIM_EPOCHS}_pgd{PGD_STEPS}_inner{INNER_EPOCHS}"
RUN_DIR = Path(RESULTS_ROOT) / "unlearnable_validation" / RUN_NAME
TABLE_DIR = RUN_DIR / "tables"
SAMPLE_DIR = RUN_DIR / "samples"
TENSOR_DIR = RUN_DIR / "tensors"
MODEL_DIR = RUN_DIR / "models"
FIGURE_DIR = RUN_DIR / "figures"
for path in [TABLE_DIR, SAMPLE_DIR, TENSOR_DIR, MODEL_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)


class CifarNormalize(nn.Module):
    def __init__(self, mean=CIFAR10_MEAN, std=CIFAR10_STD):
        super().__init__()
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return (x - self.mean.to(x.device, x.dtype)) / self.std.to(x.device, x.dtype)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_cifar_resnet18(device):
    return nn.Sequential(
        CifarNormalize(),
        get_victim_resnet18(num_classes=10, device="cpu"),
    ).to(device)


def make_loader(dataset, batch_size, shuffle, seed_offset=0):
    generator = torch.Generator().manual_seed(SEED + seed_offset)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=True,
        generator=generator if shuffle else None,
    )


def train_classifier_strong(model_fn, loader, epochs, device, lr=LEARNING_RATE):
    model = model_fn().to(device)
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        weight_decay=WEIGHT_DECAY,
        nesterov=True,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    history = []
    for epoch in range(epochs):
        loss, acc = train_one_epoch(model, loader, optimizer, criterion, device)
        scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]
        history.append({"epoch": epoch + 1, "train_loss": loss, "train_accuracy": acc, "lr": round(current_lr, 6)})
        print(f"  epoch {epoch + 1}/{epochs} loss={loss} train_acc={acc} lr={current_lr:.6f}")
    return model, history


def save_training_plot(baseline_history, victim_history, epsilon, fig_dir):
    baseline_df = pd.DataFrame(baseline_history)
    victim_df = pd.DataFrame(victim_history)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(baseline_df["epoch"], baseline_df["train_accuracy"], marker="o", label="Clean baseline train")
    ax.plot(victim_df["epoch"], victim_df["train_accuracy"], marker="s", label="Protected victim train")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Train accuracy")
    ax.set_title(f"Training accuracy curves (epsilon={epsilon})")
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(fig_dir / f"training_accuracy_eps{epsilon}.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(baseline_df["epoch"], baseline_df["train_loss"], marker="o", label="Clean baseline train")
    ax.plot(victim_df["epoch"], victim_df["train_loss"], marker="s", label="Protected victim train")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Train loss")
    ax.set_title(f"Training loss curves (epsilon={epsilon})")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(fig_dir / f"training_loss_eps{epsilon}.png", dpi=150)
    plt.close(fig)


def plot_validation_summary(df, fig_dir):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(df["epsilon"], df["protected_clean_val_accuracy"], marker="o", label="Protected val acc")
    ax.axhline(0.10, linestyle="--", color="gray", label="Random guess (CIFAR-10)")
    ax.set_xlabel("Epsilon")
    ax.set_ylabel("Clean validation accuracy")
    ax.set_title("Validation accuracy vs epsilon")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(fig_dir / "epsilon_vs_validation_accuracy.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(df["epsilon"], df["psnr"], marker="o", label="PSNR")
    ax.axhline(MIN_PSNR, linestyle="--", color="gray", label=f"PSNR threshold {MIN_PSNR}")
    ax.set_xlabel("Epsilon")
    ax.set_ylabel("PSNR (dB)")
    ax.set_title("Image quality PSNR vs epsilon")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(fig_dir / "epsilon_vs_psnr.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(df["epsilon"], df["ssim"], marker="o", color="#54A24B", label="SSIM")
    ax.axhline(MIN_SSIM, linestyle="--", color="gray", label=f"SSIM threshold {MIN_SSIM}")
    ax.set_xlabel("Epsilon")
    ax.set_ylabel("SSIM")
    ax.set_title("Image quality SSIM vs epsilon")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(fig_dir / "epsilon_vs_ssim.png", dpi=150)
    plt.close(fig)


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
setup_dirs()
print("device:", device)
print("seed:", SEED)
print("RUN_DIR:", RUN_DIR)

transform = T.Compose([T.ToTensor()])
train_full = CIFAR10(root=DATA_ROOT, train=True, download=True, transform=transform)
test_full = CIFAR10(root=DATA_ROOT, train=False, download=True, transform=transform)

needed_train = TRAIN_SUBSET_SIZE + VAL_SUBSET_SIZE
if needed_train > len(train_full):
    raise ValueError(f"TRAIN_SUBSET_SIZE + VAL_SUBSET_SIZE={needed_train} exceeds CIFAR-10 train size {len(train_full)}")

split_generator = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(train_full), generator=split_generator).tolist()
train_indices = perm[:TRAIN_SUBSET_SIZE]
val_indices = perm[TRAIN_SUBSET_SIZE:TRAIN_SUBSET_SIZE + VAL_SUBSET_SIZE]
test_indices = list(range(min(TEST_SUBSET_SIZE, len(test_full))))

train_set = Subset(train_full, train_indices)
val_set = Subset(train_full, val_indices)
test_set = Subset(test_full, test_indices)

train_loader = make_loader(train_set, BATCH_SIZE, shuffle=True, seed_offset=0)
val_loader = make_loader(val_set, BATCH_SIZE, shuffle=False)
test_loader = make_loader(test_set, BATCH_SIZE, shuffle=False)
clean_x, clean_y = collect_clean_tensors(train_loader)

print("train_protection size:", len(train_set))
print("validation size:", len(val_set))
print("test size:", len(test_set))

print("Training clean baseline once on train_protection split...")
set_seed(SEED)
baseline, baseline_history = train_classifier_strong(
    lambda: get_cifar_resnet18(device),
    train_loader,
    BASELINE_EPOCHS,
    device,
)
baseline_val_acc = evaluate(baseline, val_loader, device)
baseline_test_acc = evaluate(baseline, test_loader, device)
print("baseline_clean_val_accuracy:", baseline_val_acc)
print("baseline_clean_test_accuracy:", baseline_test_acc)

torch.save(
    {
        "model_state_dict": baseline.state_dict(),
        "model": "ResNet-18+CIFAR-normalization",
        "dataset": "CIFAR-10",
        "train_data": "clean_train_protection_split",
        "seed": SEED,
        "epochs": BASELINE_EPOCHS,
        "baseline_val_accuracy": baseline_val_acc,
        "baseline_test_accuracy": baseline_test_acc,
        "run_name": RUN_NAME,
    },
    MODEL_DIR / "baseline_clean_model.pt",
)
pd.DataFrame(baseline_history).to_csv(TABLE_DIR / "baseline_training_history.csv", index=False)

records = []
for eps in EPSILON_VALUES:
    print("\n" + "=" * 80)
    print(f"Validation run for epsilon={eps}")
    eps_name = str(eps).replace(".", "p")
    eps_tensor_dir = TENSOR_DIR / f"eps{eps_name}"
    eps_model_dir = MODEL_DIR / f"eps{eps_name}"
    eps_sample_dir = SAMPLE_DIR / f"eps{eps_name}"
    for path in [eps_tensor_dir, eps_model_dir, eps_sample_dir]:
        path.mkdir(parents=True, exist_ok=True)

    set_seed(SEED)
    noise_dict = generate_unlearnable_noise(
        model_fn=lambda: get_cifar_resnet18(device),
        train_loader=train_loader,
        epsilon=eps,
        pgd_steps=PGD_STEPS,
        inner_epochs=INNER_EPOCHS,
        device=device,
    )
    torch.save(noise_dict, eps_tensor_dir / "unlearnable_noise_dict.pt")

    protected_x = build_protected_tensors(
        "unlearnable", clean_x, clean_y, device, epsilon=eps, noise_dict=noise_dict
    )
    protected_loader = make_loader(TensorDataset(protected_x, clean_y), BATCH_SIZE, shuffle=True, seed_offset=100)

    torch.save(
        {
            "x_clean": clean_x.cpu(),
            "x_protected": protected_x.cpu(),
            "y": clean_y.cpu(),
            "epsilon": eps,
            "seed": SEED,
            "technique": "unlearnable",
            "subset_size": TRAIN_SUBSET_SIZE,
            "run_name": RUN_NAME,
        },
        eps_tensor_dir / "protected_dataset.pt",
    )

    print(f"Training victim on protected train split for epsilon={eps}...")
    set_seed(SEED)
    victim, victim_history = train_classifier_strong(
        lambda: get_cifar_resnet18(device),
        protected_loader,
        VICTIM_EPOCHS,
        device,
    )
    protected_val_acc, val_asr = compute_attack_success_rate(victim, val_loader, device)

    psnr = compute_psnr(clean_x, protected_x)
    ssim = compute_ssim(clean_x, protected_x)
    linf = compute_linf(clean_x, protected_x)
    record = {
        "technique": "unlearnable",
        "victim_model": "ResNet-18+CIFAR-normalization",
        "train_subset_size": TRAIN_SUBSET_SIZE,
        "val_subset_size": VAL_SUBSET_SIZE,
        "test_subset_size": len(test_set),
        "epsilon": eps,
        "baseline_epochs": BASELINE_EPOCHS,
        "victim_epochs": VICTIM_EPOCHS,
        "pgd_steps": PGD_STEPS,
        "inner_epochs": INNER_EPOCHS,
        "seed": SEED,
        "baseline_clean_val_accuracy": baseline_val_acc,
        "baseline_clean_test_accuracy_reference": baseline_test_acc,
        "protected_clean_val_accuracy": protected_val_acc,
        "val_accuracy_drop": round(baseline_val_acc - protected_val_acc, 4),
        "val_asr_proxy": val_asr,
        "psnr": psnr,
        "ssim": ssim,
        "linf": linf,
        "meets_quality_constraint": bool(psnr >= MIN_PSNR and ssim >= MIN_SSIM),
        "run_name": RUN_NAME,
        "run_dir": str(RUN_DIR),
    }
    print(record)
    records.append(record)

    pd.DataFrame(victim_history).to_csv(TABLE_DIR / f"victim_training_history_eps{eps_name}.csv", index=False)
    torch.save(
        {
            "model_state_dict": victim.state_dict(),
            "model": "ResNet-18+CIFAR-normalization",
            "dataset": "CIFAR-10",
            "train_data": "protected_train_protection_split",
            "epsilon": eps,
            "seed": SEED,
            "epochs": VICTIM_EPOCHS,
            "protected_val_accuracy": protected_val_acc,
            "run_name": RUN_NAME,
        },
        eps_model_dir / "victim_protected_model.pt",
    )

    sample_idx = 0
    fig = plot_before_after(clean_x[sample_idx], protected_x[sample_idx], f"unlearnable_eps{eps}", save=False)
    fig.savefig(FIGURE_DIR / f"before_after_eps{eps_name}.png", dpi=150)
    plt.close(fig)
    to_pil = T.ToPILImage()
    for idx in range(min(10, clean_x.size(0))):
        to_pil(clean_x[idx]).save(eps_sample_dir / f"original_{idx:03d}.png")
        to_pil(protected_x[idx]).save(eps_sample_dir / f"protected_{idx:03d}.png")
    save_training_plot(baseline_history, victim_history, eps, FIGURE_DIR)

    del victim, protected_x, protected_loader, noise_dict
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

val_df = pd.DataFrame(records)
val_df.to_csv(TABLE_DIR / "validation_epsilon_ablation.csv", index=False)
plot_validation_summary(val_df, FIGURE_DIR)

quality_df = val_df[val_df["meets_quality_constraint"]].copy()
if len(quality_df):
    best_row = quality_df.sort_values(["protected_clean_val_accuracy", "epsilon"], ascending=[True, True]).iloc[0]
else:
    print("WARNING: no epsilon satisfies quality constraints; selecting lowest validation accuracy.")
    best_row = val_df.sort_values(["protected_clean_val_accuracy", "epsilon"], ascending=[True, True]).iloc[0]

best_epsilon = float(best_row["epsilon"])
best_eps_name = str(best_epsilon).replace(".", "p")
print("\nSelected best_epsilon from validation:", best_epsilon)
print(best_row.to_dict())

best_model_path = MODEL_DIR / f"eps{best_eps_name}" / "victim_protected_model.pt"
checkpoint = torch.load(best_model_path, map_location=device)
best_victim = get_cifar_resnet18(device)
best_victim.load_state_dict(checkpoint["model_state_dict"])
best_test_acc, best_test_asr = compute_attack_success_rate(best_victim, test_loader, device)

final_record = dict(best_row)
final_record.update(
    {
        "selected_epsilon": best_epsilon,
        "final_protected_clean_test_accuracy": best_test_acc,
        "final_test_asr_proxy": best_test_asr,
        "final_test_accuracy_drop": round(baseline_test_acc - best_test_acc, 4),
    }
)
pd.DataFrame([final_record]).to_csv(TABLE_DIR / "final_test_selected_epsilon.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 4))
labels = ["Clean baseline", "Selected protected victim"]
values = [baseline_test_acc, best_test_acc]
bars = ax.bar(labels, values, color=["#4C78A8", "#F58518"])
ax.set_ylabel("Clean test accuracy")
ax.set_title(f"Final test accuracy drop (epsilon={best_epsilon})")
ax.set_ylim(0, max(0.55, max(values) + 0.1))
for bar, value in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.015, f"{value:.4f}", ha="center")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "final_test_accuracy_drop.png", dpi=150)
plt.close(fig)

print("\nValidation summary saved to:", TABLE_DIR / "validation_epsilon_ablation.csv")
print("Final selected test result saved to:", TABLE_DIR / "final_test_selected_epsilon.csv")
print("Figures saved to:", FIGURE_DIR)
print("Final record:")
print(final_record)
